# The core model: `PTNet`

`petrilab.petri.model` holds the single source of truth for the place/transition designs: a small `PTNet` class with weighted arcs, inhibitor arcs, read (test) arcs and optional timing/rate annotations. A **marking** is a tuple of ints ordered like `PTNet.places`.

A transition is *enabled* when every input place has at least the arc weight in tokens (plus inhibitor and read conditions); *firing* removes the input tokens and adds the output tokens.

```mermaid
flowchart LR
    H2((H2 4)) -- 2 --> react[react]
    O2((O2 2)) --> react
    react -- 2 --> H2O((H2O 0))
```

The notebook also shows the bridge to SNAKES, used to cross-check the hand-written firing rule.

In [1]:
from petrilab.petri import analysis, examples
from petrilab.petri.model import PTNet, StateSpaceLimitError
from petrilab.petri.snakes_bridge import explore, to_snakes

n = examples.water()
print(n.place_names)
print(n.initial_marking)
print(n.marking_dict(n.initial_marking))

['H2', 'O2', 'H2O']
(4, 2, 0)
{'H2': 4, 'O2': 2, 'H2O': 0}


`examples.water()` builds the net above. The initial marking is a plain tuple aligned with `place_names`; `marking_dict` turns it back into a readable `{place: tokens}` mapping.

In [2]:
m = n.initial_marking
print(n.enabled_transitions(m))
m2 = n.fire(m, "react")
print(n.marking_dict(m2))

['react']
{'H2': 2, 'O2': 1, 'H2O': 2}


`react` needs 2 `H2` and 1 `O2`, so it is enabled; firing it yields `{H2: 2, O2: 1, H2O: 2}`.

In [3]:
try:
    n.fire(n.marking(H2=1, O2=1), "react")
except ValueError as e:
    print("ValueError:", e)

ValueError: transition 'react' not enabled in {'H2': 1, 'O2': 1, 'H2O': 0}


Firing a transition that is not enabled raises `ValueError`. `net.marking(**tokens)` is a convenient constructor: unspecified places are zero.

In [4]:
bad = PTNet("x")
try:
    bad.add_transition("t", {"nope": 1})
except ValueError as e:
    print("ValueError:", e)

ValueError: unknown places ['nope'] in transition 't'


Building a net validates it: arcs referencing unknown places are rejected.

In [5]:
unbounded = examples.bounded_counter(inhibited=False)
try:
    unbounded.reachable(max_states=50)
except StateSpaceLimitError as e:
    print("StateSpaceLimitError:", e)

StateSpaceLimitError: more than 50 states


Without the inhibitor arc, `put` is always enabled and the state space is infinite. `reachable` stops with `StateSpaceLimitError` once the cap is exceeded, so exploration never runs away.

In [6]:
for net in [
    examples.water(),
    examples.bounded_counter(),
    examples.read_arc_net(),
    examples.producer_consumer(3),
    examples.dining_philosophers(3),
    examples.mutex(),
]:
    own = net.reachable()
    space = explore(to_snakes(net))
    own_dead = sum(1 for s in own.values() if not s)
    print(
        f"{net.name:28s} states own={len(own):3d} snakes={len(space):3d}  dead own={own_dead} snakes={len(analysis.dead_markings(space))}"
    )

water                        states own=  3 snakes=  3  dead own=1 snakes=1
bounded_counter              states own=  3 snakes=  3  dead own=0 snakes=0
read_arc                     states own=  3 snakes=  3  dead own=1 snakes=1
producer_consumer_3          states own=  4 snakes=  4  dead own=0 snakes=0
philosophers_3_naive         states own= 14 snakes= 14  dead own=1 snakes=1
mutex                        states own=  8 snakes=  8  dead own=0 snakes=0


Each net is compiled to a SNAKES net with `to_snakes` and explored with `explore`. The state count and dead-marking count agree with the hand-written engine, which validates the firing rule against an independent implementation.

In [7]:
print(examples.water().incidence())

[[-2], [-1], [2]]


`incidence()` returns the matrix `C[place][transition] = post - pre`. Rows follow `places` (`H2`, `O2`, `H2O`), columns follow transitions (`react`): `-2`, `-1`, `+2`. It is the basis for the invariant analysis in the next notebook.